In [0]:
%run ../read_params

In [0]:
%run ../utils

In [0]:
%run ./schemas

In [0]:
failed_runs_in_a_row = 0

pokedex_lists = []

for i in range(1,100):
    print(f'Attempting to extract pokedex {i}...')
    try:
        s_start_time = time.time()

        dex_url = f'https://pokeapi.co/api/v2/pokedex/{i}/'
        dex_df = api_extraction(dex_url, dex_schema)
        
        s_end_time = time.time()
        time.sleep(s_end_time - s_start_time)

        t_dex_df = dex_df.select(
            col('id'), 
            col('name'), 
            col('is_main_series'), 
            col('region.name').alias('region'),
            col('version_groups'),
            col('pokemon_entries')
        )

        pokedex_lists.append(t_dex_df)

        print(f"Successfully Extracted Pokedex {i}")

        failed_runs_in_a_row = 0

    except Exception as e:
        failed_runs_in_a_row += 1
        print(f'Failed to extract pokedex {i} - {e}')

    finally:
        if failed_runs_in_a_row >= 2:
            if len(pokedex_lists) != 0:
                pokedex_df = reduce(DataFrame.union, pokedex_lists)

            print('Stopping Extractions due to multiple non-existent pokedex ids!')
            break


pokedex_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.extracted_pokedexes")

In [0]:
pokedex_version_groups = (
    pokedex_df
    .withColumn('version_groups', explode(col('version_groups')))
    .select(
        col('id'),
        col('version_groups.name').alias('version_group'),
        col('version_groups.url').alias('version_group_url')
    )
)

pokedex_version_groups.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.pokedex_version_groups")

In [0]:
version_group_list = [[row['id'], row['version_group'], row['version_group_url']] for row in pokedex_version_groups.select("id", "version_group", "version_group_url").collect()]

version_lists = []

for id, version_group, version_group_url in version_group_list:
    print(f'Extracting versions from {version_group}...')
    base_dict = {
        'version_group_id': id
    }

    version_group_df = api_extraction(version_group_url, version_group_schema, base_dict)

    version_lists.append(version_group_df)

    print(f'Completed extraction of versions from {version_group}!')

versions_df = reduce(DataFrame.union, version_lists)
versions_df.write.format('delta').mode("overwrite").option('overwriteSchema', 'true').saveAsTable(f"{STAGING_DATABASE_PREFIX}.game_versions")